# 03 — XGBoost grouped validation

本 notebook 使用与 Random Forest 完全相同的 37,724 个像元、557 个
polygon、206 个空间组、`sample_weight` 和五个外层 folds，对 XGBoost
进行特征组合比较、nested grouped validation、有限调参和最终模型训练。

关键原则：

- 不重新抽样，不重新分配 RF 的外层 folds。
- `group_uid` 在所有训练与验证划分中保持隔离。
- 特征组合与参数选择仅发生在外层训练数据内部。
- validation metrics 仍使用等 polygon 总权重。
- 使用分块读取、`n_jobs=2`、逐配置 checkpoint 和逐 fold 输出，降低 Colab
  内存风险并支持中断续跑。
- 原始 CSV 和 RF 结果目录均只读；XGBoost 使用独立结果文件夹。

## 正确打开方式

在 Colab 中选择 **File → Open notebook → Upload**，上传整个
`03_XGBoost_grouped_validation.ipynb`。不要把 `.ipynb` 的 JSON 原文粘贴到
Python 单元。请从上到下逐个运行；每个单元前均说明作用、输入、输出和耗时。

本 notebook 默认使用稳定的 `RUN_NAME`。若 Colab 中断，重新连接后使用相同
`RUN_NAME` 从头运行，已完成的 checkpoint 会自动复用。只有在修改实验配置后，
才应更换 `RUN_NAME`。

## 流程

1. 挂载 Drive 并读取 RF manifest。
2. 复用 RF 保存的训练行、权重和外层 folds。
3. 从原始 71 万行 CSV 中分块提取所需的 37,724 行特征。
4. 使用固定 XGBoost 比较五套 feature stacks。
5. 在每个 outer fold 内重新选择 feature stack 和 XGBoost 参数。
6. 汇总 OOF 结果并与 RF 做同折比较。
7. 使用全部 Bentong 数据训练最终 XGBoost 并保存 manifest。

### 0.1 挂载 Google Drive

- **作用：** 让 Colab 访问原始像元 CSV、RF 结果和新的 XGBoost 结果目录。
- **输入：** Google 账号授权。
- **输出：** `<DURIAN_DATA_ROOT>/` 可访问。
- **耗时：** 首次运行需要授权，通常很短。

In [ ]:
from pathlib import Path
import os


def _find_repository_root(start: Path) -> Path:
    current = start.resolve()
    while current.parent != current:
        if (current / "README.md").exists() and (current / "code").exists():
            return current
        current = current.parent
    raise RuntimeError("Run this notebook from within the repository tree.")


REPO_ROOT = _find_repository_root(Path.cwd())
DATA_ROOT = Path(
    os.environ.get("DURIAN_DATA_ROOT", REPO_ROOT / "private_data")
).expanduser().resolve()
REPO_OUTPUT_ROOT = Path(
    os.environ.get("DURIAN_OUTPUT_ROOT", REPO_ROOT / "outputs" / "runs")
).expanduser().resolve()
REPO_OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)

print("Private data root:", DATA_ROOT)
print("Run output root:", REPO_OUTPUT_ROOT)


### 0.2 检查并导入 Python 库

- **作用：** 加载 XGBoost、表格、绘图、分组验证和模型保存依赖；仅在缺少 XGBoost 时安装。
- **输入：** Colab Python 环境。
- **输出：** 后续单元所需的全部库及版本信息。
- **耗时：** 数秒；首次安装 XGBoost 时可能需要 1–2 分钟。

In [ ]:
import importlib.util
import subprocess
import sys

if importlib.util.find_spec('xgboost') is None:
    subprocess.check_call([
        sys.executable, '-m', 'pip', 'install', '--quiet', 'xgboost'
    ])

import gc
import hashlib
import json
import os
import platform
import time
import warnings
from datetime import datetime, timezone
from pathlib import Path

import joblib
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
import sklearn
import xgboost as xgb

from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    classification_report,
    confusion_matrix,
    f1_score,
    precision_recall_fscore_support,
)
from sklearn.model_selection import StratifiedGroupKFold
from xgboost import XGBClassifier

warnings.filterwarnings('once')
sns.set_theme(style='whitegrid', context='notebook')

print('Python:', sys.version.split()[0])
print('pandas:', pd.__version__)
print('scikit-learn:', sklearn.__version__)
print('XGBoost:', xgb.__version__)

### 0.3 设置 Drive 路径和可续跑的 RUN_NAME

- **作用：** 指定原始 CSV、已完成的 RF 结果目录和本次 XGBoost 输出目录。
- **输入：** 用户给定的 RF 结果路径所对应的 Drive 文件夹。
- **输出：** 所有输入及输出路径变量。
- **耗时：** 立即完成。

In [ ]:
DRIVE_ROOT = DATA_ROOT

INPUT_CSV = DRIVE_ROOT / 'Bentong_pixel_samples_2025_v1.csv'
RF_RESULT_DIR = (
    DRIVE_ROOT / 'RF_grouped_validation_20260623_153912_UTC'
)

# 中断续跑时保持不变；修改实验配置后请改成新的名称。
RUN_NAME = 'XGBoost_grouped_validation_20260624_run01'
OUTPUT_DIR = REPO_OUTPUT_ROOT / RUN_NAME

TABLE_DIR = OUTPUT_DIR / 'tables'
FIGURE_DIR = OUTPUT_DIR / 'figures'
MODEL_DIR = OUTPUT_DIR / 'models'
METADATA_DIR = OUTPUT_DIR / 'metadata'
CHECKPOINT_DIR = OUTPUT_DIR / 'checkpoints'

VERIFY_INPUT_SHA256 = True
CSV_CHUNK_SIZE = 100_000

required_paths = [
    INPUT_CSV,
    RF_RESULT_DIR / 'metadata' / 'model_manifest.json',
    RF_RESULT_DIR / 'metadata' / 'nested_overall_metrics.json',
    RF_RESULT_DIR / 'tables' / 'training_rows_used.csv',
    RF_RESULT_DIR / 'tables' / 'fold_assignments.csv',
    RF_RESULT_DIR / 'tables' / 'oof_polygon_predictions.csv',
]
missing_paths = [str(path) for path in required_paths if not path.exists()]
if missing_paths:
    raise FileNotFoundError(
        'Required input files are missing:\n' + '\n'.join(missing_paths)
    )

print('Input CSV:', INPUT_CSV)
print('RF result folder:', RF_RESULT_DIR)
print('XGBoost output folder:', OUTPUT_DIR)

## 1. 读取 RF 设计并锁定 XGBoost 配置

### 1.1 读取 RF manifest 和 nested metrics

- **作用：** 继承 RF 的类别、feature stacks、样本数量、fold 数和输入数据哈希。
- **输入：** RF `model_manifest.json` 与 `nested_overall_metrics.json`。
- **输出：** `rf_manifest`、`rf_nested_metrics`。
- **耗时：** 很短。

In [ ]:
with open(
    RF_RESULT_DIR / 'metadata' / 'model_manifest.json',
    'r',
    encoding='utf-8',
) as file:
    rf_manifest = json.load(file)

with open(
    RF_RESULT_DIR / 'metadata' / 'nested_overall_metrics.json',
    'r',
    encoding='utf-8',
) as file:
    rf_nested_metrics = json.load(file)

print('RF model rows:', rf_manifest['model_rows'])
print('RF samples:', rf_manifest['sample_count'])
print('RF groups:', rf_manifest['group_count'])
print('RF selected feature set:', rf_manifest['selected_feature_set'])
print('RF polygon Durian F1:', rf_nested_metrics['polygon_durian_f1'])

### 1.2 锁定类别、features 和 grouped CV 参数

- **作用：** 直接继承 RF 的定义，避免模型比较因字段或类别编码不同而失真。
- **输入：** `rf_manifest`。
- **输出：** 七类映射、五套 feature stacks、五个 outer folds 和三个 inner folds。
- **耗时：** 立即完成。

In [ ]:
RANDOM_SEED = int(rf_manifest['random_seed'])
N_SPLITS = int(rf_manifest['n_splits'])
INNER_SPLITS = int(rf_manifest['inner_splits'])
N_JOBS = 2
WORKFLOW_VERSION = 'xgb_grouped_v1_20260624'
TRAINING_WEIGHT_METHOD = (
    'RF polygon sample_weight multiplied by training-split class '
    'frequency factor, then normalized to mean 1'
)

CLASS_TO_ID = {
    str(name): int(class_id)
    for name, class_id in rf_manifest['class_to_id'].items()
}
ID_TO_CLASS = {class_id: name for name, class_id in CLASS_TO_ID.items()}
CLASS_IDS = sorted(ID_TO_CLASS)
CLASS_NAMES = [ID_TO_CLASS[class_id] for class_id in CLASS_IDS]
DURIAN_ID = CLASS_TO_ID['Durian']

FEATURE_SETS = {
    str(name): list(features)
    for name, features in rf_manifest['feature_sets'].items()
}
FULL_BANDS = FEATURE_SETS['FULL']

EXPECTED_MODEL_ROWS = int(rf_manifest['model_rows'])
EXPECTED_SAMPLE_COUNT = int(rf_manifest['sample_count'])
EXPECTED_GROUP_COUNT = int(rf_manifest['group_count'])

print('Classes:', CLASS_TO_ID)
print('Feature sets:', {name: len(value) for name, value in FEATURE_SETS.items()})
print('Outer / inner folds:', N_SPLITS, '/', INNER_SPLITS)
print('XGBoost n_jobs:', N_JOBS)

### 1.3 设置 baseline XGBoost 与有限参数候选

- **作用：** 使用透明的小型候选集控制计算量；不使用 early stopping，避免额外的非分组内部验证。
- **输入：** 人工锁定的 XGBoost 参数。
- **输出：** baseline 参数和五个候选配置。
- **耗时：** 立即完成。

In [ ]:
BASELINE_XGB_PARAMS = {
    'n_estimators': 350,
    'learning_rate': 0.05,
    'max_depth': 6,
    'min_child_weight': 2.0,
    'subsample': 0.8,
    'colsample_bytree': 0.8,
    'gamma': 0.0,
    'reg_alpha': 0.0,
    'reg_lambda': 1.0,
}

XGB_CANDIDATES = [
    {
        'candidate_id': 'X0',
        'n_estimators': 300,
        'learning_rate': 0.05,
        'max_depth': 4,
        'min_child_weight': 1.0,
        'subsample': 0.8,
        'colsample_bytree': 0.8,
        'gamma': 0.0,
        'reg_alpha': 0.0,
        'reg_lambda': 1.0,
    },
    {
        'candidate_id': 'X1',
        'n_estimators': 400,
        'learning_rate': 0.05,
        'max_depth': 6,
        'min_child_weight': 2.0,
        'subsample': 0.8,
        'colsample_bytree': 0.8,
        'gamma': 0.0,
        'reg_alpha': 0.0,
        'reg_lambda': 1.0,
    },
    {
        'candidate_id': 'X2',
        'n_estimators': 500,
        'learning_rate': 0.03,
        'max_depth': 4,
        'min_child_weight': 2.0,
        'subsample': 0.85,
        'colsample_bytree': 0.85,
        'gamma': 0.0,
        'reg_alpha': 0.0,
        'reg_lambda': 1.0,
    },
    {
        'candidate_id': 'X3',
        'n_estimators': 350,
        'learning_rate': 0.04,
        'max_depth': 8,
        'min_child_weight': 3.0,
        'subsample': 0.8,
        'colsample_bytree': 0.8,
        'gamma': 0.0,
        'reg_alpha': 0.05,
        'reg_lambda': 2.0,
    },
    {
        'candidate_id': 'X4',
        'n_estimators': 400,
        'learning_rate': 0.04,
        'max_depth': 6,
        'min_child_weight': 5.0,
        'subsample': 0.8,
        'colsample_bytree': 0.8,
        'gamma': 0.1,
        'reg_alpha': 0.1,
        'reg_lambda': 5.0,
    },
]

print('Candidate count:', len(XGB_CANDIDATES))
display(pd.DataFrame(XGB_CANDIDATES))

### 1.4 创建输出目录并锁定 run signature

- **作用：** 确保中断续跑时不会把不同配置的 checkpoint 混在一起。
- **输入：** 路径、RF manifest、feature sets 和 XGBoost 参数。
- **输出：** 独立结果目录及 `run_config.json`。
- **耗时：** 很短。

In [ ]:
def sha256_file(path, chunk_size=1024 * 1024):
    digest = hashlib.sha256()
    with open(path, 'rb') as file:
        while True:
            chunk = file.read(chunk_size)
            if not chunk:
                break
            digest.update(chunk)
    return digest.hexdigest()


training_rows_path = RF_RESULT_DIR / 'tables' / 'training_rows_used.csv'
fold_assignments_path = RF_RESULT_DIR / 'tables' / 'fold_assignments.csv'

run_signature_payload = {
    'source_rf_manifest_sha256': sha256_file(
        RF_RESULT_DIR / 'metadata' / 'model_manifest.json'
    ),
    'training_rows_sha256': sha256_file(training_rows_path),
    'fold_assignments_sha256': sha256_file(fold_assignments_path),
    'random_seed': RANDOM_SEED,
    'n_splits': N_SPLITS,
    'inner_splits': INNER_SPLITS,
    'workflow_version': WORKFLOW_VERSION,
    'training_weight_method': TRAINING_WEIGHT_METHOD,
    'xgboost_version': xgb.__version__,
    'scikit_learn_version': sklearn.__version__,
    'feature_sets': FEATURE_SETS,
    'baseline_xgb_params': BASELINE_XGB_PARAMS,
    'xgb_candidates': XGB_CANDIDATES,
}
run_signature = hashlib.sha256(
    json.dumps(
        run_signature_payload,
        sort_keys=True,
        ensure_ascii=False,
    ).encode('utf-8')
).hexdigest()

for folder in [
    OUTPUT_DIR,
    TABLE_DIR,
    FIGURE_DIR,
    MODEL_DIR,
    METADATA_DIR,
    CHECKPOINT_DIR,
]:
    folder.mkdir(parents=True, exist_ok=True)

run_config_path = METADATA_DIR / 'run_config.json'
if run_config_path.exists():
    with open(run_config_path, 'r', encoding='utf-8') as file:
        existing_run_config = json.load(file)
    if existing_run_config.get('run_signature') != run_signature:
        raise ValueError(
            'RUN_NAME already contains checkpoints from a different '
            'configuration. Change RUN_NAME before continuing.'
        )
    print('Existing compatible run found; checkpoints may be reused.')
else:
    with open(run_config_path, 'w', encoding='utf-8') as file:
        json.dump(
            {
                'created_utc': datetime.now(timezone.utc).isoformat(),
                'run_name': RUN_NAME,
                'run_signature': run_signature,
                **run_signature_payload,
            },
            file,
            indent=2,
            ensure_ascii=False,
        )

print('Run signature:', run_signature)
print('Output directory:', OUTPUT_DIR)

## 2. 复用 RF 训练行并低内存提取特征

### 2.1 读取 RF 保存的训练行和 fold assignments

- **作用：** 获得完全相同的像元、polygon 权重与外层验证 folds。
- **输入：** RF `training_rows_used.csv` 和 `fold_assignments.csv`。
- **输出：** `training_rows`、`fold_assignments`。
- **耗时：** 很短。

In [ ]:
training_rows = pd.read_csv(
    training_rows_path,
    dtype={
        'pixel_uid': 'string',
        'sample_uid': 'string',
        'group_uid': 'string',
        'class_lv2': 'string',
    },
)
fold_assignments = pd.read_csv(
    fold_assignments_path,
    dtype={
        'sample_uid': 'string',
        'group_uid': 'string',
        'class_lv2': 'string',
    },
)

for column in ['class_id', 'fold_id']:
    training_rows[column] = pd.to_numeric(
        training_rows[column], errors='raise'
    ).astype(int)
training_rows['sample_weight'] = pd.to_numeric(
    training_rows['sample_weight'], errors='raise'
).astype(float)

for column in ['class_id', 'fold_id']:
    fold_assignments[column] = pd.to_numeric(
        fold_assignments[column], errors='raise'
    ).astype(int)

print('Training rows:', f'{len(training_rows):,}')
print('Fold assignment rows:', len(fold_assignments))
display(training_rows.head())

### 2.2 审计复用数据和空间隔离

- **作用：** 确认像元唯一、557/206 数量一致、polygon 权重为 1，且 group 不跨 fold。
- **输入：** `training_rows`、`fold_assignments`。
- **输出：** 审计摘要和 `training_rows_reused.csv`。
- **耗时：** 较短。

In [ ]:
if len(training_rows) != EXPECTED_MODEL_ROWS:
    raise ValueError(
        f'Expected {EXPECTED_MODEL_ROWS} model rows, found '
        f'{len(training_rows)}.'
    )
if training_rows['pixel_uid'].isna().any():
    raise ValueError('pixel_uid contains null values.')
if training_rows['pixel_uid'].duplicated().any():
    raise ValueError('pixel_uid is not unique in training_rows_used.csv.')
if training_rows['sample_uid'].nunique() != EXPECTED_SAMPLE_COUNT:
    raise ValueError('Unexpected sample count in reused training rows.')
if training_rows['group_uid'].nunique() != EXPECTED_GROUP_COUNT:
    raise ValueError('Unexpected group count in reused training rows.')
if sorted(training_rows['class_id'].unique()) != CLASS_IDS:
    raise ValueError('Reused training rows do not contain all locked classes.')
if sorted(training_rows['fold_id'].unique()) != list(range(N_SPLITS)):
    raise ValueError('Unexpected outer fold IDs.')

polygon_weight_totals = training_rows.groupby('sample_uid')[
    'sample_weight'
].sum()
if not np.allclose(polygon_weight_totals.to_numpy(), 1.0):
    raise ValueError('Reused polygon sample weights do not sum to 1.')

if training_rows.groupby('group_uid')['fold_id'].nunique().max() != 1:
    raise ValueError('At least one group_uid crosses outer folds.')

fold_check = training_rows[
    ['sample_uid', 'group_uid', 'class_id', 'class_lv2', 'fold_id']
].drop_duplicates('sample_uid')
merged_fold_check = fold_check.merge(
    fold_assignments,
    on='sample_uid',
    how='outer',
    suffixes=('_training', '_assignment'),
    indicator=True,
    validate='one_to_one',
)
if not (merged_fold_check['_merge'] == 'both').all():
    raise ValueError('training_rows and fold_assignments sample sets differ.')
for column in ['group_uid', 'class_id', 'class_lv2', 'fold_id']:
    if not (
        merged_fold_check[f'{column}_training'].astype(str)
        == merged_fold_check[f'{column}_assignment'].astype(str)
    ).all():
        raise ValueError(f'Fold metadata mismatch in column: {column}')

reused_summary = (
    training_rows.groupby(['class_id', 'class_lv2'])
    .agg(
        pixel_rows=('pixel_uid', 'size'),
        samples=('sample_uid', 'nunique'),
        groups=('group_uid', 'nunique'),
    )
    .reset_index()
    .sort_values('class_id')
)
training_rows.to_csv(
    TABLE_DIR / 'training_rows_reused.csv', index=False
)
reused_summary.to_csv(
    TABLE_DIR / 'reused_training_summary.csv', index=False
)

display(reused_summary)
print('Group cross-fold audit: passed')
print('Polygon total-weight range:',
      polygon_weight_totals.min(), polygon_weight_totals.max())

### 2.3 验证原始 CSV 哈希

- **作用：** 确认 XGBoost 使用的原始 CSV 与 RF manifest 记录的是同一个文件。
- **输入：** 原始 CSV 和 RF manifest 中的 SHA256。
- **输出：** `actual_input_sha256`。
- **耗时：** 需要顺序读取 CSV；通常几十秒至数分钟。可通过配置关闭。

In [ ]:
if VERIFY_INPUT_SHA256:
    hash_start = time.time()
    actual_input_sha256 = sha256_file(INPUT_CSV)
    print(
        f'CSV SHA256 runtime: {(time.time() - hash_start) / 60:.1f} minutes'
    )
    if actual_input_sha256 != rf_manifest['input_csv_sha256']:
        raise ValueError(
            'Input CSV SHA256 does not match the RF manifest. '
            'Do not compare models until the input file is corrected.'
        )
else:
    actual_input_sha256 = rf_manifest['input_csv_sha256']
    print('SHA256 verification skipped by configuration.')

print('Input CSV SHA256:', actual_input_sha256)

### 2.4 分块提取 RF 实际使用像元的 26 个特征

- **作用：** 避免把 712,772 行完整 CSV 同时载入内存；每次只读取一个 chunk，并保留目标 pixel_uid。
- **输入：** 原始像元 CSV、RF 的 37,724 个 pixel_uid。
- **输出：** 仅含目标像元的 `feature_rows`。
- **耗时：** 通常数十秒至数分钟，取决于 Drive 读取速度。

In [ ]:
selected_pixel_ids = set(training_rows['pixel_uid'].astype(str))
feature_parts = []
scanned_rows = 0
retained_rows = 0

read_columns = ['pixel_uid'] + FULL_BANDS
for chunk_index, chunk in enumerate(
    pd.read_csv(
        INPUT_CSV,
        usecols=read_columns,
        dtype={'pixel_uid': 'string'},
        chunksize=CSV_CHUNK_SIZE,
        low_memory=False,
    ),
    start=1,
):
    scanned_rows += len(chunk)
    keep_mask = chunk['pixel_uid'].astype(str).isin(selected_pixel_ids)
    retained_chunk = chunk.loc[keep_mask].copy()
    retained_rows += len(retained_chunk)
    if not retained_chunk.empty:
        feature_parts.append(retained_chunk)

    print(
        f'Chunk {chunk_index}: scanned={scanned_rows:,}, '
        f'retained={retained_rows:,}'
    )
    del chunk, retained_chunk
    gc.collect()

if not feature_parts:
    raise ValueError('No RF training pixel_uid values were found in the CSV.')

feature_rows = pd.concat(feature_parts, ignore_index=True)
feature_parts.clear()
gc.collect()

if feature_rows['pixel_uid'].duplicated().any():
    raise ValueError('Selected feature rows contain duplicate pixel_uid values.')
if len(feature_rows) != EXPECTED_MODEL_ROWS:
    raise ValueError(
        f'Expected {EXPECTED_MODEL_ROWS} selected feature rows, found '
        f'{len(feature_rows)}.'
    )

print('Total CSV rows scanned:', f'{scanned_rows:,}')
print('Selected feature rows:', f'{len(feature_rows):,}')

### 2.5 合并固定训练设计与特征并做最终审计

- **作用：** 构建 XGBoost 唯一建模表，并把 predictors 转成 float32 节省内存。
- **输入：** `training_rows` 与 `feature_rows`。
- **输出：** `model_df`、`sample_table` 和数据审计 JSON。
- **耗时：** 较短。

In [ ]:
model_df = training_rows.merge(
    feature_rows,
    on='pixel_uid',
    how='left',
    validate='one_to_one',
)
del feature_rows
gc.collect()

for feature in FULL_BANDS:
    model_df[feature] = pd.to_numeric(
        model_df[feature], errors='coerce'
    ).astype('float32')
model_df[FULL_BANDS] = model_df[FULL_BANDS].replace(
    [np.inf, -np.inf], np.nan
)

missing_feature_rows = model_df[FULL_BANDS].isna().any(axis=1)
if missing_feature_rows.any():
    raise ValueError(
        f'{missing_feature_rows.sum()} reused RF rows have missing XGBoost '
        'predictors.'
    )

sample_table = (
    model_df[
        ['sample_uid', 'group_uid', 'class_id', 'class_lv2', 'fold_id']
    ]
    .drop_duplicates('sample_uid')
    .sort_values('sample_uid')
    .reset_index(drop=True)
)

data_audit = {
    'input_csv_rows_scanned': int(scanned_rows),
    'model_rows': int(len(model_df)),
    'sample_count': int(model_df['sample_uid'].nunique()),
    'group_count': int(model_df['group_uid'].nunique()),
    'outer_fold_count': int(model_df['fold_id'].nunique()),
    'predictor_count': len(FULL_BANDS),
    'memory_mb': float(model_df.memory_usage(deep=True).sum() / 1e6),
}
with open(
    METADATA_DIR / 'xgb_data_audit.json', 'w', encoding='utf-8'
) as file:
    json.dump(data_audit, file, indent=2, ensure_ascii=False)

print(json.dumps(data_audit, indent=2))
display(model_df.head())

## 3. XGBoost、指标与 checkpoint 辅助函数

### 3.1 定义原子写入和 checkpoint 工具

- **作用：** 先写临时文件再替换正式文件，避免 Colab 中断后留下被误认为完成的半文件。
- **输入：** Python 对象或 dataframe。
- **输出：** JSON/CSV checkpoint 工具函数。
- **耗时：** 这里只定义函数。

In [ ]:
def json_default(value):
    if isinstance(value, np.generic):
        return value.item()
    if isinstance(value, Path):
        return str(value)
    raise TypeError(f'Object is not JSON serializable: {type(value)}')


def atomic_write_json(payload, output_path):
    output_path = Path(output_path)
    output_path.parent.mkdir(parents=True, exist_ok=True)
    temporary_path = output_path.with_suffix(output_path.suffix + '.tmp')
    with open(temporary_path, 'w', encoding='utf-8') as file:
        json.dump(
            payload,
            file,
            indent=2,
            ensure_ascii=False,
            default=json_default,
        )
    os.replace(temporary_path, output_path)


def atomic_write_csv(frame, output_path, index=False):
    output_path = Path(output_path)
    output_path.parent.mkdir(parents=True, exist_ok=True)
    temporary_path = output_path.with_suffix(output_path.suffix + '.tmp')
    frame.to_csv(temporary_path, index=index)
    os.replace(temporary_path, output_path)


def load_json(path):
    with open(path, 'r', encoding='utf-8') as file:
        return json.load(file)

### 3.2 定义 XGBoost 训练权重和拟合函数

- **作用：** 保留等 polygon 权重，并在每个训练 split 内计算类别平衡因子；权重均值归一为 1。
- **输入：** 训练 dataframe、features 和 XGBoost 参数。
- **输出：** 已拟合的 `XGBClassifier`。
- **耗时：** 这里只定义函数。

In [ ]:
def clean_xgb_params(params):
    return {
        key: value
        for key, value in params.items()
        if key != 'candidate_id'
    }


def make_xgb_training_weights(train_frame):
    class_row_counts = train_frame['class_id'].value_counts()
    if set(class_row_counts.index.astype(int)) != set(CLASS_IDS):
        raise ValueError('A training split is missing at least one class.')

    class_factors = {
        int(class_id): len(train_frame)
        / (len(CLASS_IDS) * int(row_count))
        for class_id, row_count in class_row_counts.items()
    }
    weights = (
        train_frame['sample_weight'].to_numpy(dtype='float64')
        * train_frame['class_id'].map(class_factors).to_numpy(dtype='float64')
    )

    # XGBoost regularization uses absolute weight scale, so preserve ratios
    # while normalizing the mean weight to 1.
    weights *= len(weights) / weights.sum()
    return weights.astype('float32')


def fit_xgb(train_frame, features, params, seed_offset=0):
    model = XGBClassifier(
        **clean_xgb_params(params),
        objective='multi:softprob',
        num_class=len(CLASS_IDS),
        eval_metric='mlogloss',
        tree_method='hist',
        importance_type='gain',
        n_jobs=N_JOBS,
        random_state=RANDOM_SEED + seed_offset,
        verbosity=0,
    )
    model.fit(
        train_frame[features],
        train_frame['class_id'],
        sample_weight=make_xgb_training_weights(train_frame),
        verbose=False,
    )
    return model

### 3.3 定义像元预测与 polygon 概率聚合

- **作用：** 产生七类概率，并以 polygon 内平均概率确定 polygon 预测类别。
- **输入：** 模型、验证 dataframe 和 predictors。
- **输出：** 像元级与 polygon 级预测表。
- **耗时：** 这里只定义函数。

In [ ]:
def make_pixel_predictions(model, frame, features):
    predicted_ids = model.predict(frame[features]).astype(int)
    probabilities = model.predict_proba(frame[features])

    output_columns = [
        'pixel_uid', 'sample_uid', 'group_uid', 'class_lv2',
        'class_id', 'fold_id', 'sample_weight',
    ]
    output = frame[output_columns].copy()
    output['pred_class_id'] = predicted_ids
    output['pred_class_name'] = output['pred_class_id'].map(ID_TO_CLASS)

    for class_id in CLASS_IDS:
        output[f'prob_{class_id}'] = 0.0
    for column_index, class_id in enumerate(model.classes_.astype(int)):
        output[f'prob_{class_id}'] = probabilities[:, column_index]
    return output


def aggregate_polygon_predictions(pixel_predictions):
    probability_columns = [f'prob_{class_id}' for class_id in CLASS_IDS]
    aggregations = {
        'group_uid': 'first',
        'class_lv2': 'first',
        'class_id': 'first',
        'fold_id': 'first',
    }
    aggregations.update({column: 'mean' for column in probability_columns})

    polygon_predictions = (
        pixel_predictions.groupby('sample_uid', as_index=False)
        .agg(aggregations)
    )
    probability_matrix = polygon_predictions[
        probability_columns
    ].to_numpy()
    polygon_predictions['pred_class_id'] = np.asarray(CLASS_IDS)[
        probability_matrix.argmax(axis=1)
    ]
    polygon_predictions['pred_class_name'] = polygon_predictions[
        'pred_class_id'
    ].map(ID_TO_CLASS)
    return polygon_predictions

### 3.4 定义评价指标和统一评估函数

- **作用：** 使用与 RF 相同的 accuracy、balanced accuracy、macro F1 和 Durian 指标。
- **输入：** 真实类别、预测类别和可选的 polygon 权重。
- **输出：** 像元级与 polygon 级 metrics。
- **耗时：** 这里只定义函数。

In [ ]:
def metric_dictionary(y_true, y_pred, sample_weight=None):
    precision, recall, durian_f1, _ = precision_recall_fscore_support(
        y_true,
        y_pred,
        labels=[DURIAN_ID],
        average=None,
        sample_weight=sample_weight,
        zero_division=0,
    )
    return {
        'accuracy': accuracy_score(
            y_true, y_pred, sample_weight=sample_weight
        ),
        'balanced_accuracy': balanced_accuracy_score(
            y_true, y_pred, sample_weight=sample_weight
        ),
        'macro_f1': f1_score(
            y_true,
            y_pred,
            labels=CLASS_IDS,
            average='macro',
            sample_weight=sample_weight,
            zero_division=0,
        ),
        'durian_precision': float(precision[0]),
        'durian_recall': float(recall[0]),
        'durian_f1': float(durian_f1[0]),
    }


def evaluate_model(model, validation_frame, features):
    pixel_predictions = make_pixel_predictions(
        model, validation_frame, features
    )
    polygon_predictions = aggregate_polygon_predictions(
        pixel_predictions
    )
    pixel_metrics = metric_dictionary(
        pixel_predictions['class_id'],
        pixel_predictions['pred_class_id'],
        sample_weight=pixel_predictions['sample_weight'],
    )
    polygon_metrics = metric_dictionary(
        polygon_predictions['class_id'],
        polygon_predictions['pred_class_id'],
    )
    combined_metrics = {
        **{f'pixel_{key}': value for key, value in pixel_metrics.items()},
        **{
            f'polygon_{key}': value
            for key, value in polygon_metrics.items()
        },
    }
    return combined_metrics, pixel_predictions, polygon_predictions

### 3.5 定义报告和混淆矩阵输出

- **作用：** 统一保存分类报告、CSV confusion matrix 和 PNG 图。
- **输入：** OOF 预测结果。
- **输出：** 可复现的表格和图件。
- **耗时：** 这里只定义函数。

In [ ]:
def report_frame(y_true, y_pred, weights=None):
    report = classification_report(
        y_true,
        y_pred,
        labels=CLASS_IDS,
        target_names=CLASS_NAMES,
        sample_weight=weights,
        zero_division=0,
        output_dict=True,
    )
    return pd.DataFrame(report).T


def save_confusion_figure(
    y_true,
    y_pred,
    title,
    output_name,
    weights=None,
):
    matrix = confusion_matrix(
        y_true,
        y_pred,
        labels=CLASS_IDS,
        sample_weight=weights,
    )
    matrix_frame = pd.DataFrame(
        matrix,
        index=CLASS_NAMES,
        columns=CLASS_NAMES,
    )
    atomic_write_csv(
        matrix_frame,
        TABLE_DIR / f'{output_name}.csv',
        index=True,
    )

    plt.figure(figsize=(9, 7))
    sns.heatmap(matrix_frame, annot=True, fmt='.1f', cmap='Blues')
    plt.title(title)
    plt.xlabel('Predicted class')
    plt.ylabel('Reference class')
    plt.tight_layout()
    plt.savefig(
        FIGURE_DIR / f'{output_name}.png',
        dpi=180,
        bbox_inches='tight',
    )
    plt.show()

### 3.6 定义有效的 inner grouped folds

- **作用：** 为每个 outer training split 寻找所有 inner folds 均含七类的分组方案，并保存复用。
- **输入：** outer training polygon 表。
- **输出：** inner fold assignments 和所用随机种子。
- **耗时：** 这里只定义函数。

In [ ]:
def build_valid_grouped_assignments(
    samples,
    n_splits,
    base_seed,
    max_attempts=1000,
):
    samples = samples.reset_index(drop=True).copy()
    dummy_x = np.zeros((len(samples), 1))
    best_fold_ids = None
    best_seed = None
    best_score = np.inf

    for attempt in range(max_attempts):
        seed = base_seed + attempt
        splitter = StratifiedGroupKFold(
            n_splits=n_splits,
            shuffle=True,
            random_state=seed,
        )
        fold_ids = np.full(len(samples), -1, dtype=int)
        for fold_id, (_, validation_indices) in enumerate(
            splitter.split(
                dummy_x,
                samples['class_id'],
                groups=samples['group_uid'],
            )
        ):
            fold_ids[validation_indices] = fold_id

        test_frame = samples.assign(inner_fold_id=fold_ids)
        class_counts = pd.crosstab(
            test_frame['inner_fold_id'],
            test_frame['class_id'],
        ).reindex(
            index=range(n_splits),
            columns=CLASS_IDS,
            fill_value=0,
        )
        if (class_counts == 0).any().any():
            continue

        group_counts = (
            test_frame.groupby(['inner_fold_id', 'class_id'])[
                'group_uid'
            ]
            .nunique()
            .unstack(fill_value=0)
            .reindex(
                index=range(n_splits),
                columns=CLASS_IDS,
                fill_value=0,
            )
        )
        expected = 1.0 / n_splits
        class_proportions = class_counts.div(
            class_counts.sum(axis=0), axis=1
        )
        group_proportions = group_counts.div(
            group_counts.sum(axis=0), axis=1
        )
        score = (
            ((class_proportions - expected) ** 2).to_numpy().sum()
            + ((group_proportions - expected) ** 2).to_numpy().sum()
        )
        if score < best_score:
            best_fold_ids = fold_ids.copy()
            best_seed = seed
            best_score = score

    if best_fold_ids is None:
        raise ValueError(
            f'Could not create valid {n_splits}-fold grouped assignments.'
        )

    assignments = samples[
        ['sample_uid', 'group_uid', 'class_id', 'class_lv2']
    ].copy()
    assignments['inner_fold_id'] = best_fold_ids
    assignments['split_seed'] = best_seed

    if assignments.groupby('group_uid')['inner_fold_id'].nunique().max() != 1:
        raise AssertionError('An inner group crosses folds.')
    return assignments, best_seed, best_score


def get_or_create_inner_assignments(outer_fold_id, outer_train_samples):
    output_path = (
        CHECKPOINT_DIR
        / 'nested'
        / f'outer_{outer_fold_id}'
        / 'inner_fold_assignments.csv'
    )
    if output_path.exists():
        assignments = pd.read_csv(
            output_path,
            dtype={
                'sample_uid': 'string',
                'group_uid': 'string',
                'class_lv2': 'string',
            },
        )
        if set(assignments['sample_uid'].astype(str)) != set(
            outer_train_samples['sample_uid'].astype(str)
        ):
            raise ValueError(
                f'Stale inner assignments for outer fold {outer_fold_id}.'
            )
        return assignments

    assignments, split_seed, balance_score = (
        build_valid_grouped_assignments(
            outer_train_samples,
            n_splits=INNER_SPLITS,
            base_seed=RANDOM_SEED + 1000 + outer_fold_id * 1000,
        )
    )
    atomic_write_csv(assignments, output_path, index=False)
    print(
        f'Outer {outer_fold_id}: inner seed={split_seed}, '
        f'balance={balance_score:.6f}'
    )
    return assignments

### 3.7 定义 inner CV 配置评分函数

- **作用：** 在预先保存的 inner grouped folds 上评价一个 feature/参数配置，训练后立即释放模型。
- **输入：** outer training 像元、inner assignments、features 和参数。
- **输出：** inner fold metrics 与平均指标。
- **耗时：** 这里只定义函数。

In [ ]:
SELECTION_METRICS = [
    'polygon_durian_f1_mean',
    'polygon_macro_f1_mean',
    'pixel_durian_f1_mean',
]


def score_configuration_on_inner_folds(
    outer_train_pixels,
    inner_assignments,
    features,
    params,
    seed_base,
):
    fold_records = []
    for inner_fold_id in range(INNER_SPLITS):
        validation_uids = set(
            inner_assignments.loc[
                inner_assignments['inner_fold_id'] == inner_fold_id,
                'sample_uid',
            ].astype(str)
        )
        train_frame = outer_train_pixels.loc[
            ~outer_train_pixels['sample_uid'].astype(str).isin(
                validation_uids
            )
        ]
        validation_frame = outer_train_pixels.loc[
            outer_train_pixels['sample_uid'].astype(str).isin(
                validation_uids
            )
        ]

        if set(train_frame['group_uid']) & set(validation_frame['group_uid']):
            raise AssertionError('Inner training/validation group leakage.')

        model = fit_xgb(
            train_frame,
            features,
            params,
            seed_offset=seed_base + inner_fold_id,
        )
        metrics, pixel_predictions, polygon_predictions = evaluate_model(
            model, validation_frame, features
        )
        fold_records.append({
            'inner_fold_id': inner_fold_id,
            **metrics,
        })
        del model, pixel_predictions, polygon_predictions
        gc.collect()

    fold_frame = pd.DataFrame(fold_records)
    return {
        'polygon_durian_f1_mean': float(
            fold_frame['polygon_durian_f1'].mean()
        ),
        'polygon_macro_f1_mean': float(
            fold_frame['polygon_macro_f1'].mean()
        ),
        'pixel_durian_f1_mean': float(
            fold_frame['pixel_durian_f1'].mean()
        ),
        'pixel_macro_f1_mean': float(
            fold_frame['pixel_macro_f1'].mean()
        ),
        'inner_fold_metrics': fold_records,
    }

## 4. 固定 XGBoost 的 feature-stack 比较

### 4.1 在 RF 外层 folds 上运行五套 feature stacks

- **作用：** 固定 baseline XGBoost，只改变 predictors；每个 feature × fold 单独 checkpoint。
- **输入：** `model_df`、RF outer folds 和 `FEATURE_SETS`。
- **输出：** 25 个 fold-level score checkpoints。
- **耗时：** 首个明显耗时步骤；中断后可续跑。

In [ ]:
feature_fold_records = []
feature_checkpoint_dir = CHECKPOINT_DIR / 'feature_comparison'
feature_checkpoint_dir.mkdir(parents=True, exist_ok=True)

for feature_set_name, features in FEATURE_SETS.items():
    print(f'Feature set: {feature_set_name} ({len(features)} features)')
    for fold_id in range(N_SPLITS):
        checkpoint_path = (
            feature_checkpoint_dir
            / f'{feature_set_name}_fold_{fold_id}.json'
        )
        if checkpoint_path.exists():
            record = load_json(checkpoint_path)
            print(f'  fold {fold_id}: reused')
        else:
            train_frame = model_df.loc[model_df['fold_id'] != fold_id]
            validation_frame = model_df.loc[
                model_df['fold_id'] == fold_id
            ]
            model = fit_xgb(
                train_frame,
                features,
                BASELINE_XGB_PARAMS,
                seed_offset=100 + fold_id,
            )
            metrics, pixel_predictions, polygon_predictions = evaluate_model(
                model, validation_frame, features
            )
            record = {
                'feature_set': feature_set_name,
                'feature_count': len(features),
                'fold_id': fold_id,
                **metrics,
            }
            atomic_write_json(record, checkpoint_path)
            del model, pixel_predictions, polygon_predictions
            gc.collect()
            print(f'  fold {fold_id}: completed')
        feature_fold_records.append(record)

feature_fold_metrics = pd.DataFrame(feature_fold_records)
atomic_write_csv(
    feature_fold_metrics,
    TABLE_DIR / 'feature_set_fold_metrics.csv',
    index=False,
)
display(feature_fold_metrics.head())

### 4.2 汇总并选择 full-data feature stack

- **作用：** 按 polygon Durian F1、polygon macro F1、pixel Durian F1 依次排序。
- **输入：** 25 个 feature-fold metrics。
- **输出：** `SELECTED_FEATURE_SET`、summary CSV 和比较图。
- **耗时：** 较短。

In [ ]:
feature_summary = (
    feature_fold_metrics.groupby('feature_set')
    .agg(
        feature_count=('feature_count', 'first'),
        polygon_durian_f1_mean=('polygon_durian_f1', 'mean'),
        polygon_durian_f1_std=('polygon_durian_f1', 'std'),
        polygon_macro_f1_mean=('polygon_macro_f1', 'mean'),
        polygon_macro_f1_std=('polygon_macro_f1', 'std'),
        pixel_durian_f1_mean=('pixel_durian_f1', 'mean'),
        pixel_macro_f1_mean=('pixel_macro_f1', 'mean'),
        polygon_accuracy_mean=('polygon_accuracy', 'mean'),
    )
    .reset_index()
    .sort_values(SELECTION_METRICS, ascending=False)
    .reset_index(drop=True)
)
SELECTED_FEATURE_SET = str(feature_summary.loc[0, 'feature_set'])
SELECTED_FEATURES = FEATURE_SETS[SELECTED_FEATURE_SET]

atomic_write_csv(
    feature_summary,
    TABLE_DIR / 'feature_set_summary.csv',
    index=False,
)

plot_data = feature_summary.melt(
    id_vars=['feature_set'],
    value_vars=[
        'polygon_durian_f1_mean',
        'polygon_macro_f1_mean',
    ],
    var_name='metric',
    value_name='score',
)
plt.figure(figsize=(10, 5))
sns.barplot(data=plot_data, x='feature_set', y='score', hue='metric')
plt.ylim(0, 1)
plt.title('XGBoost grouped feature-stack comparison')
plt.xlabel('Feature set')
plt.ylabel('Mean 5-fold score')
plt.tight_layout()
plt.savefig(
    FIGURE_DIR / 'feature_set_comparison.png',
    dpi=180,
    bbox_inches='tight',
)
plt.show()

print('Selected full-data feature set:', SELECTED_FEATURE_SET)
display(feature_summary)

## 5. Nested grouped feature selection 与 XGBoost 调参

### 5.1 运行可续跑的 nested grouped validation

- **作用：** 每个 outer fold 内先选 feature stack，再选参数，最后只在未见 outer fold 评价一次。
- **输入：** RF outer folds、有效 inner folds、五套 feature stacks 和五个候选。
- **输出：** 逐配置 checkpoints、逐 outer-fold OOF 预测与 metrics。
- **耗时：** 全 notebook 最耗时；所有细分任务均可续跑。

In [ ]:
nested_feature_records = []
nested_candidate_records = []
nested_outer_records = []
nested_selected_records = []

nested_start = time.time()
for outer_fold_id in range(N_SPLITS):
    print(f'\n===== Outer fold {outer_fold_id + 1}/{N_SPLITS} =====')
    outer_dir = CHECKPOINT_DIR / 'nested' / f'outer_{outer_fold_id}'
    outer_dir.mkdir(parents=True, exist_ok=True)

    outer_train_pixels = model_df.loc[
        model_df['fold_id'] != outer_fold_id
    ]
    outer_validation_pixels = model_df.loc[
        model_df['fold_id'] == outer_fold_id
    ]
    outer_train_samples = sample_table.loc[
        sample_table['fold_id'] != outer_fold_id
    ]
    if set(outer_train_pixels['group_uid']) & set(
        outer_validation_pixels['group_uid']
    ):
        raise AssertionError('Outer training/validation group leakage.')

    inner_assignments = get_or_create_inner_assignments(
        outer_fold_id, outer_train_samples
    )

    # Stage A: feature-stack selection inside outer training only.
    fold_feature_rows = []
    for feature_set_name, features in FEATURE_SETS.items():
        checkpoint_path = outer_dir / f'feature_{feature_set_name}.json'
        if checkpoint_path.exists():
            scores = load_json(checkpoint_path)
            print(f'  feature {feature_set_name}: reused')
        else:
            scores = score_configuration_on_inner_folds(
                outer_train_pixels,
                inner_assignments,
                features,
                BASELINE_XGB_PARAMS,
                seed_base=10_000 + outer_fold_id * 100,
            )
            atomic_write_json(scores, checkpoint_path)
            print(f'  feature {feature_set_name}: completed')
        row = {
            'outer_fold_id': outer_fold_id,
            'feature_set': feature_set_name,
            'feature_count': len(features),
            **{
                key: value
                for key, value in scores.items()
                if key != 'inner_fold_metrics'
            },
        }
        fold_feature_rows.append(row)
        nested_feature_records.append(row)

    fold_feature_frame = (
        pd.DataFrame(fold_feature_rows)
        .sort_values(SELECTION_METRICS, ascending=False)
        .reset_index(drop=True)
    )
    fold_feature_set = str(fold_feature_frame.loc[0, 'feature_set'])
    fold_features = FEATURE_SETS[fold_feature_set]
    print('  selected feature set:', fold_feature_set)

    # Stage B: XGBoost tuning inside the same outer training data.
    fold_candidate_rows = []
    for candidate in XGB_CANDIDATES:
        candidate_id = candidate['candidate_id']
        checkpoint_path = outer_dir / f'candidate_{candidate_id}.json'
        if checkpoint_path.exists():
            scores = load_json(checkpoint_path)
            print(f'  candidate {candidate_id}: reused')
        else:
            scores = score_configuration_on_inner_folds(
                outer_train_pixels,
                inner_assignments,
                fold_features,
                candidate,
                seed_base=20_000 + outer_fold_id * 100,
            )
            atomic_write_json(scores, checkpoint_path)
            print(f'  candidate {candidate_id}: completed')
        row = {
            'outer_fold_id': outer_fold_id,
            'selected_feature_set': fold_feature_set,
            'candidate_id': candidate_id,
            **{
                key: value
                for key, value in scores.items()
                if key != 'inner_fold_metrics'
            },
        }
        fold_candidate_rows.append(row)
        nested_candidate_records.append(row)

    fold_candidate_frame = (
        pd.DataFrame(fold_candidate_rows)
        .sort_values(SELECTION_METRICS, ascending=False)
        .reset_index(drop=True)
    )
    best_candidate_id = str(
        fold_candidate_frame.loc[0, 'candidate_id']
    )
    best_candidate = next(
        candidate
        for candidate in XGB_CANDIDATES
        if candidate['candidate_id'] == best_candidate_id
    )
    print('  selected candidate:', best_candidate_id)

    nested_selected_records.append({
        'outer_fold_id': outer_fold_id,
        'selected_feature_set': fold_feature_set,
        'selected_feature_count': len(fold_features),
        **best_candidate,
    })

    outer_metrics_path = outer_dir / 'outer_metrics.json'
    outer_pixel_path = outer_dir / 'oof_pixel_predictions.csv'
    outer_polygon_path = outer_dir / 'oof_polygon_predictions.csv'
    complete_path = outer_dir / 'complete.json'

    if (
        complete_path.exists()
        and outer_metrics_path.exists()
        and outer_pixel_path.exists()
        and outer_polygon_path.exists()
    ):
        outer_record = load_json(outer_metrics_path)
        print('  outer evaluation: reused')
    else:
        outer_model = fit_xgb(
            outer_train_pixels,
            fold_features,
            best_candidate,
            seed_offset=30_000 + outer_fold_id,
        )
        metrics, pixel_predictions, polygon_predictions = evaluate_model(
            outer_model,
            outer_validation_pixels,
            fold_features,
        )
        outer_record = {
            'outer_fold_id': outer_fold_id,
            'selected_feature_set': fold_feature_set,
            'candidate_id': best_candidate_id,
            **metrics,
        }
        atomic_write_csv(pixel_predictions, outer_pixel_path, index=False)
        atomic_write_csv(
            polygon_predictions, outer_polygon_path, index=False
        )
        atomic_write_json(outer_record, outer_metrics_path)
        atomic_write_json(
            {
                'complete': True,
                'completed_utc': datetime.now(timezone.utc).isoformat(),
                'run_signature': run_signature,
            },
            complete_path,
        )
        del outer_model, pixel_predictions, polygon_predictions
        gc.collect()
        print('  outer evaluation: completed')

    nested_outer_records.append(outer_record)
    del outer_train_pixels, outer_validation_pixels, inner_assignments
    gc.collect()

print(
    f'Nested runtime this session: '
    f'{(time.time() - nested_start) / 60:.1f} minutes'
)

nested_feature_scores = pd.DataFrame(nested_feature_records)
nested_candidate_scores = pd.DataFrame(nested_candidate_records)
nested_fold_metrics = pd.DataFrame(nested_outer_records)
nested_selected = pd.DataFrame(nested_selected_records)

atomic_write_csv(
    nested_feature_scores,
    TABLE_DIR / 'nested_inner_feature_set_scores.csv',
    index=False,
)
atomic_write_csv(
    nested_candidate_scores,
    TABLE_DIR / 'nested_inner_candidate_scores.csv',
    index=False,
)
atomic_write_csv(
    nested_fold_metrics,
    TABLE_DIR / 'nested_outer_fold_metrics.csv',
    index=False,
)
atomic_write_csv(
    nested_selected,
    TABLE_DIR / 'nested_selected_candidates.csv',
    index=False,
)
display(nested_fold_metrics)

### 5.2 低内存合并五个 outer-fold OOF 文件

- **作用：** 逐文件、逐 chunk 写出完整 OOF CSV；内存仅加载计算指标所需列。
- **输入：** 五个 outer fold 的像元和 polygon prediction CSV。
- **输出：** 完整 OOF CSV 与轻量内存评价表。
- **耗时：** 较短；不会使用大规模 `pd.concat`。

In [ ]:
def stream_csv_files(input_paths, output_path, chunksize=50_000):
    output_path = Path(output_path)
    temporary_path = output_path.with_suffix(output_path.suffix + '.tmp')
    if temporary_path.exists():
        temporary_path.unlink()

    total_rows = 0
    for input_path in input_paths:
        for chunk in pd.read_csv(input_path, chunksize=chunksize):
            chunk.to_csv(
                temporary_path,
                mode='w' if total_rows == 0 else 'a',
                header=total_rows == 0,
                index=False,
            )
            total_rows += len(chunk)
            del chunk
            gc.collect()
    if total_rows == 0:
        raise ValueError(f'No rows written to {output_path}.')
    os.replace(temporary_path, output_path)
    return total_rows


outer_dirs = [
    CHECKPOINT_DIR / 'nested' / f'outer_{fold_id}'
    for fold_id in range(N_SPLITS)
]
pixel_fold_paths = [
    folder / 'oof_pixel_predictions.csv' for folder in outer_dirs
]
polygon_fold_paths = [
    folder / 'oof_polygon_predictions.csv' for folder in outer_dirs
]
missing_oof = [
    str(path)
    for path in pixel_fold_paths + polygon_fold_paths
    if not path.exists()
]
if missing_oof:
    raise FileNotFoundError('Missing OOF files:\n' + '\n'.join(missing_oof))

oof_pixel_path = TABLE_DIR / 'oof_pixel_predictions.csv'
oof_polygon_path = TABLE_DIR / 'oof_polygon_predictions.csv'
pixel_rows_written = stream_csv_files(pixel_fold_paths, oof_pixel_path)
polygon_rows_written = stream_csv_files(
    polygon_fold_paths, oof_polygon_path
)

oof_pixel_predictions = pd.read_csv(
    oof_pixel_path,
    usecols=[
        'pixel_uid', 'sample_uid', 'class_id',
        'pred_class_id', 'sample_weight',
    ],
    dtype={
        'pixel_uid': 'string',
        'sample_uid': 'string',
        'class_id': 'int16',
        'pred_class_id': 'int16',
        'sample_weight': 'float32',
    },
)
oof_polygon_predictions = pd.read_csv(
    oof_polygon_path,
    dtype={
        'sample_uid': 'string',
        'group_uid': 'string',
        'class_lv2': 'string',
        'class_id': 'int16',
        'pred_class_id': 'int16',
    },
)

if pixel_rows_written != EXPECTED_MODEL_ROWS:
    raise ValueError('Unexpected combined OOF pixel row count.')
if polygon_rows_written != EXPECTED_SAMPLE_COUNT:
    raise ValueError('Unexpected combined OOF polygon row count.')
if oof_pixel_predictions['pixel_uid'].duplicated().any():
    raise ValueError('OOF pixel predictions contain duplicate pixel_uid.')
if oof_polygon_predictions['sample_uid'].duplicated().any():
    raise ValueError('OOF polygon predictions contain duplicate sample_uid.')

print('OOF pixel rows:', f'{pixel_rows_written:,}')
print('OOF polygon rows:', f'{polygon_rows_written:,}')

### 5.3 计算 nested OOF 总体指标和报告

- **作用：** 合并所有未见 outer-fold 预测，生成与 RF 同定义的正式内部验证结果。
- **输入：** OOF 像元和 polygon 预测。
- **输出：** metrics JSON、分类报告和两套混淆矩阵。
- **耗时：** 数秒至数十秒。

In [ ]:
overall_pixel_metrics = metric_dictionary(
    oof_pixel_predictions['class_id'],
    oof_pixel_predictions['pred_class_id'],
    sample_weight=oof_pixel_predictions['sample_weight'],
)
overall_polygon_metrics = metric_dictionary(
    oof_polygon_predictions['class_id'],
    oof_polygon_predictions['pred_class_id'],
)
nested_feature_counts = {
    str(name): int(count)
    for name, count in nested_selected[
        'selected_feature_set'
    ].value_counts().items()
}
nested_overall_metrics = {
    'full_data_selected_feature_set': SELECTED_FEATURE_SET,
    'outer_fold_selected_feature_set_counts': nested_feature_counts,
    **{
        f'pixel_{key}': value
        for key, value in overall_pixel_metrics.items()
    },
    **{
        f'polygon_{key}': value
        for key, value in overall_polygon_metrics.items()
    },
}
atomic_write_json(
    nested_overall_metrics,
    METADATA_DIR / 'nested_overall_metrics.json',
)

pixel_report = report_frame(
    oof_pixel_predictions['class_id'],
    oof_pixel_predictions['pred_class_id'],
    weights=oof_pixel_predictions['sample_weight'],
)
polygon_report = report_frame(
    oof_polygon_predictions['class_id'],
    oof_polygon_predictions['pred_class_id'],
)
atomic_write_csv(
    pixel_report,
    TABLE_DIR / 'oof_pixel_classification_report.csv',
    index=True,
)
atomic_write_csv(
    polygon_report,
    TABLE_DIR / 'oof_polygon_classification_report.csv',
    index=True,
)

save_confusion_figure(
    oof_pixel_predictions['class_id'],
    oof_pixel_predictions['pred_class_id'],
    'XGBoost nested grouped CV — weighted pixel confusion matrix',
    'oof_pixel_confusion_matrix',
    weights=oof_pixel_predictions['sample_weight'],
)
save_confusion_figure(
    oof_polygon_predictions['class_id'],
    oof_polygon_predictions['pred_class_id'],
    'XGBoost nested grouped CV — polygon confusion matrix',
    'oof_polygon_confusion_matrix',
)

print(json.dumps(nested_overall_metrics, indent=2))
display(polygon_report)

## 6. 选择最终参数并训练完整 Bentong XGBoost

### 6.1 在固定 outer folds 上评价最终候选参数

- **作用：** nested CV 已完成性能估计后，用全部 Bentong folds 为部署模型选定一组最终参数。
- **输入：** `SELECTED_FEATURES`、五个候选和 RF outer folds。
- **输出：** 每个 candidate × fold 的 checkpoint。
- **耗时：** 较耗时；中断后可续跑。

In [ ]:
final_checkpoint_dir = CHECKPOINT_DIR / 'final_candidate_selection'
final_checkpoint_dir.mkdir(parents=True, exist_ok=True)
final_fold_records = []

for candidate in XGB_CANDIDATES:
    candidate_id = candidate['candidate_id']
    print(f'Final candidate: {candidate_id}')
    for fold_id in range(N_SPLITS):
        checkpoint_path = (
            final_checkpoint_dir
            / f'{candidate_id}_fold_{fold_id}.json'
        )
        if checkpoint_path.exists():
            record = load_json(checkpoint_path)
            print(f'  fold {fold_id}: reused')
        else:
            train_frame = model_df.loc[model_df['fold_id'] != fold_id]
            validation_frame = model_df.loc[
                model_df['fold_id'] == fold_id
            ]
            model = fit_xgb(
                train_frame,
                SELECTED_FEATURES,
                candidate,
                seed_offset=40_000 + fold_id,
            )
            metrics, pixel_predictions, polygon_predictions = evaluate_model(
                model, validation_frame, SELECTED_FEATURES
            )
            record = {
                'candidate_id': candidate_id,
                'fold_id': fold_id,
                **metrics,
            }
            atomic_write_json(record, checkpoint_path)
            del model, pixel_predictions, polygon_predictions
            gc.collect()
            print(f'  fold {fold_id}: completed')
        final_fold_records.append(record)

final_candidate_fold_metrics = pd.DataFrame(final_fold_records)
atomic_write_csv(
    final_candidate_fold_metrics,
    TABLE_DIR / 'final_candidate_fold_metrics.csv',
    index=False,
)

### 6.2 汇总并选择最终 XGBoost 参数

- **作用：** 使用与 RF 相同的指标优先级确定部署候选。
- **输入：** candidate-fold metrics。
- **输出：** `FINAL_XGB_PARAMS` 和 `final_candidate_scores.csv`。
- **耗时：** 较短。

In [ ]:
final_candidate_scores = (
    final_candidate_fold_metrics.groupby('candidate_id')
    .agg(
        polygon_durian_f1_mean=('polygon_durian_f1', 'mean'),
        polygon_macro_f1_mean=('polygon_macro_f1', 'mean'),
        pixel_durian_f1_mean=('pixel_durian_f1', 'mean'),
        pixel_macro_f1_mean=('pixel_macro_f1', 'mean'),
    )
    .reset_index()
    .sort_values(SELECTION_METRICS, ascending=False)
    .reset_index(drop=True)
)
FINAL_CANDIDATE_ID = str(
    final_candidate_scores.loc[0, 'candidate_id']
)
FINAL_XGB_PARAMS = next(
    candidate
    for candidate in XGB_CANDIDATES
    if candidate['candidate_id'] == FINAL_CANDIDATE_ID
)
atomic_write_csv(
    final_candidate_scores,
    TABLE_DIR / 'final_candidate_scores.csv',
    index=False,
)

print('Selected feature set:', SELECTED_FEATURE_SET)
print('Selected final candidate:', FINAL_CANDIDATE_ID)
display(final_candidate_scores)

### 6.3 使用全部 Bentong 数据训练并保存最终 XGBoost

- **作用：** 训练下一阶段可调用的完整模型；若模型 JSON 已存在则直接载入。
- **输入：** 全部 37,724 个像元、最终 feature stack 和参数。
- **输出：** XGBoost JSON、UBJ、joblib 模型和 bundle。
- **耗时：** 中等；首次运行需要训练一次。

In [ ]:
final_json_path = MODEL_DIR / 'xgb_final_model.json'
final_ubj_path = MODEL_DIR / 'xgb_final_model.ubj'

if final_json_path.exists():
    final_model = XGBClassifier()
    final_model.load_model(str(final_json_path))
    print('Existing final XGBoost model loaded.')
else:
    final_model = fit_xgb(
        model_df,
        SELECTED_FEATURES,
        FINAL_XGB_PARAMS,
        seed_offset=50_000,
    )
    print('Final XGBoost model trained and saved.')

# Always ensure both portable native formats exist, including after resume.
final_model.save_model(str(final_json_path))
final_model.save_model(str(final_ubj_path))

joblib.dump(final_model, MODEL_DIR / 'xgb_final_model.joblib')
model_bundle = {
    'model': final_model,
    'feature_set': SELECTED_FEATURE_SET,
    'predictor_bands': SELECTED_FEATURES,
    'class_to_id': CLASS_TO_ID,
    'id_to_class': ID_TO_CLASS,
    'xgb_params': clean_xgb_params(FINAL_XGB_PARAMS),
    'candidate_id': FINAL_CANDIDATE_ID,
    'random_seed': RANDOM_SEED,
    'source_rf_result_dir': str(RF_RESULT_DIR),
    'run_signature': run_signature,
}
joblib.dump(model_bundle, MODEL_DIR / 'xgb_final_bundle.joblib')

### 6.4 保存最终 feature importance

- **作用：** 输出基于 gain 的 XGBoost feature importance。
- **输入：** 最终模型和 selected predictors。
- **输出：** CSV 与 PNG。
- **耗时：** 较短。

In [ ]:
feature_importance = (
    pd.DataFrame({
        'feature': SELECTED_FEATURES,
        'importance_gain': final_model.feature_importances_,
    })
    .sort_values('importance_gain', ascending=False)
    .reset_index(drop=True)
)
atomic_write_csv(
    feature_importance,
    TABLE_DIR / 'feature_importance.csv',
    index=False,
)

plt.figure(figsize=(9, max(5, len(feature_importance) * 0.28)))
sns.barplot(
    data=feature_importance,
    y='feature',
    x='importance_gain',
    color='#F59E0B',
)
plt.title('Final XGBoost feature importance (gain)')
plt.xlabel('Normalized gain importance')
plt.ylabel('Feature')
plt.tight_layout()
plt.savefig(
    FIGURE_DIR / 'feature_importance.png',
    dpi=180,
    bbox_inches='tight',
)
plt.show()
display(feature_importance)

## 7. 使用相同 OOF 样本比较 RF 与 XGBoost

### 7.1 生成 RF–XGBoost 指标和 polygon 配对比较

- **作用：** 在完全相同的 557 个 polygon 上比较两套 nested OOF 结果。
- **输入：** RF 与 XGBoost nested metrics、polygon OOF predictions。
- **输出：** 统一指标表、配对正确性表、分歧摘要和比较图。
- **耗时：** 较短。

In [ ]:
comparison_metric_names = [
    'pixel_accuracy',
    'pixel_balanced_accuracy',
    'pixel_macro_f1',
    'pixel_durian_f1',
    'polygon_accuracy',
    'polygon_balanced_accuracy',
    'polygon_macro_f1',
    'polygon_durian_f1',
]
rf_xgb_metrics = pd.DataFrame([
    {
        'model': 'Random Forest',
        **{
            metric: rf_nested_metrics[metric]
            for metric in comparison_metric_names
        },
    },
    {
        'model': 'XGBoost',
        **{
            metric: nested_overall_metrics[metric]
            for metric in comparison_metric_names
        },
    },
])
atomic_write_csv(
    rf_xgb_metrics,
    TABLE_DIR / 'rf_xgb_nested_metrics_comparison.csv',
    index=False,
)

rf_polygon_oof = pd.read_csv(
    RF_RESULT_DIR / 'tables' / 'oof_polygon_predictions.csv',
    usecols=['sample_uid', 'class_id', 'pred_class_id'],
    dtype={
        'sample_uid': 'string',
        'class_id': 'int16',
        'pred_class_id': 'int16',
    },
).rename(columns={'pred_class_id': 'rf_pred_class_id'})
xgb_polygon_oof = oof_polygon_predictions[
    ['sample_uid', 'class_id', 'pred_class_id']
].rename(columns={'pred_class_id': 'xgb_pred_class_id'})

paired_polygon_comparison = rf_polygon_oof.merge(
    xgb_polygon_oof,
    on='sample_uid',
    how='outer',
    suffixes=('_rf', '_xgb'),
    validate='one_to_one',
    indicator=True,
)
if not (paired_polygon_comparison['_merge'] == 'both').all():
    raise ValueError('RF and XGBoost OOF polygon sample sets differ.')
if not (
    paired_polygon_comparison['class_id_rf']
    == paired_polygon_comparison['class_id_xgb']
).all():
    raise ValueError('RF and XGBoost reference class IDs differ.')

paired_polygon_comparison['class_id'] = paired_polygon_comparison[
    'class_id_rf'
]
paired_polygon_comparison['rf_correct'] = (
    paired_polygon_comparison['rf_pred_class_id']
    == paired_polygon_comparison['class_id']
)
paired_polygon_comparison['xgb_correct'] = (
    paired_polygon_comparison['xgb_pred_class_id']
    == paired_polygon_comparison['class_id']
)
paired_polygon_comparison['correctness_pattern'] = (
    paired_polygon_comparison['rf_correct'].astype(int).astype(str)
    + '_'
    + paired_polygon_comparison['xgb_correct'].astype(int).astype(str)
)
atomic_write_csv(
    paired_polygon_comparison.drop(columns=['_merge']),
    TABLE_DIR / 'rf_xgb_paired_polygon_predictions.csv',
    index=False,
)
disagreement_summary = (
    paired_polygon_comparison.groupby('correctness_pattern')
    .size()
    .rename('polygon_count')
    .reset_index()
)
atomic_write_csv(
    disagreement_summary,
    TABLE_DIR / 'rf_xgb_correctness_disagreement.csv',
    index=False,
)

plot_metrics = rf_xgb_metrics.melt(
    id_vars='model',
    value_vars=[
        'polygon_accuracy',
        'polygon_macro_f1',
        'polygon_durian_f1',
    ],
    var_name='metric',
    value_name='score',
)
plt.figure(figsize=(10, 5))
sns.barplot(data=plot_metrics, x='metric', y='score', hue='model')
plt.ylim(0, 1)
plt.title('Nested grouped CV: RF vs XGBoost')
plt.xlabel('')
plt.ylabel('Score')
plt.xticks(rotation=20)
plt.tight_layout()
plt.savefig(
    FIGURE_DIR / 'rf_xgb_nested_metric_comparison.png',
    dpi=180,
    bbox_inches='tight',
)
plt.show()

display(rf_xgb_metrics)
display(disagreement_summary)

## 8. 保存 manifest、README 和结果清单

### 8.1 写入最终可复现记录

- **作用：** 记录输入哈希、RF 来源、软件版本、参数、features、nested metrics 和全部文件。
- **输入：** 本次完整运行状态。
- **输出：** model manifest、README、inventory 和 completion marker。
- **耗时：** 较短。

In [ ]:
manifest = {
    'created_utc': datetime.now(timezone.utc).isoformat(),
    'run_name': RUN_NAME,
    'run_signature': run_signature,
    'input_csv': str(INPUT_CSV),
    'input_csv_sha256': actual_input_sha256,
    'source_rf_result_directory': str(RF_RESULT_DIR),
    'source_rf_manifest_sha256': run_signature_payload[
        'source_rf_manifest_sha256'
    ],
    'source_training_rows_sha256': run_signature_payload[
        'training_rows_sha256'
    ],
    'source_fold_assignments_sha256': run_signature_payload[
        'fold_assignments_sha256'
    ],
    'output_directory': str(OUTPUT_DIR),
    'model_rows': int(len(model_df)),
    'sample_count': int(model_df['sample_uid'].nunique()),
    'group_count': int(model_df['group_uid'].nunique()),
    'n_splits': N_SPLITS,
    'inner_splits': INNER_SPLITS,
    'random_seed': RANDOM_SEED,
    'n_jobs': N_JOBS,
    'class_to_id': CLASS_TO_ID,
    'feature_sets': FEATURE_SETS,
    'selected_feature_set': SELECTED_FEATURE_SET,
    'selected_predictor_bands': SELECTED_FEATURES,
    'final_candidate_id': FINAL_CANDIDATE_ID,
    'final_xgb_params': clean_xgb_params(FINAL_XGB_PARAMS),
    'workflow_version': WORKFLOW_VERSION,
    'training_weight_method': TRAINING_WEIGHT_METHOD,
    'nested_overall_metrics': nested_overall_metrics,
    'rf_nested_overall_metrics': rf_nested_metrics,
    'software': {
        'python': sys.version,
        'platform': platform.platform(),
        'numpy': np.__version__,
        'pandas': pd.__version__,
        'scikit_learn': sklearn.__version__,
        'xgboost': xgb.__version__,
        'joblib': joblib.__version__,
    },
}
atomic_write_json(manifest, METADATA_DIR / 'model_manifest.json')

readme_text = f'''XGBoost grouped validation results
====================================
Input CSV: {INPUT_CSV}
Source RF result: {RF_RESULT_DIR}
Reused model rows: {len(model_df)}
Samples: {model_df['sample_uid'].nunique()}
Groups: {model_df['group_uid'].nunique()}
Selected feature set: {SELECTED_FEATURE_SET}
Selected candidate: {FINAL_CANDIDATE_ID}
Predictor count: {len(SELECTED_FEATURES)}

Validation design
-----------------
RF outer folds and training rows are reused exactly.
Feature-stack and parameter selection are repeated inside outer training data.
Checkpoints allow interrupted Colab sessions to resume with the same RUN_NAME.

Key files
---------
models/xgb_final_model.json
models/xgb_final_model.ubj
models/xgb_final_bundle.joblib
metadata/model_manifest.json
metadata/nested_overall_metrics.json
tables/feature_set_summary.csv
tables/nested_outer_fold_metrics.csv
tables/oof_pixel_predictions.csv
tables/oof_polygon_predictions.csv
tables/final_candidate_scores.csv
tables/feature_importance.csv
tables/rf_xgb_nested_metrics_comparison.csv
figures/oof_pixel_confusion_matrix.png
figures/oof_polygon_confusion_matrix.png
figures/feature_importance.png
figures/rf_xgb_nested_metric_comparison.png
'''
(OUTPUT_DIR / 'README.txt').write_text(readme_text, encoding='utf-8')

inventory = []
for path in sorted(OUTPUT_DIR.rglob('*')):
    if path.is_file():
        inventory.append({
            'relative_path': str(path.relative_to(OUTPUT_DIR)),
            'size_bytes': int(path.stat().st_size),
        })
inventory_frame = pd.DataFrame(inventory)
atomic_write_csv(
    inventory_frame,
    TABLE_DIR / 'output_inventory.csv',
    index=False,
)
atomic_write_json(
    {
        'complete': True,
        'completed_utc': datetime.now(timezone.utc).isoformat(),
        'run_signature': run_signature,
    },
    METADATA_DIR / 'RUN_COMPLETE.json',
)

print('XGBoost workflow completed successfully.')
print('Results folder:', OUTPUT_DIR)
display(inventory_frame)